# Atelier Python Session 7 (suite) : visualiser des données complexes avec scikit-learn

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-7/Atelier%20Python%20Session%207%20-%20Data%20vis%202%20-%20scikit-learn.ipynb)

Dans le premier notebook, nos données n'avaient que deux colonnes (x et y) : facile de les dessiner. Mais en SHS, les
données ont souvent **beaucoup** de dimensions : les réponses à un questionnaire de 50 questions, la fréquence de
milliers de mots dans chaque document d'un corpus, les pixels d'une image... Comment visualiser cela ?

Voici la chaîne de traitement (le *pipeline*) que nous allons suivre, avec le package
[scikit-learn](https://scikit-learn.org/) (`sklearn`), la référence de l'apprentissage automatique en Python :

1. **Données** : on part d'un tableau de nombres (si nos données sont des textes ou des images, il faut d'abord les
   transformer en nombres : c'est la **vectorisation**) ;
2. **Prétraitement** : on met toutes les colonnes à la même échelle ;
3. **Réduction de dimension** : on résume les nombreuses colonnes en seulement deux, en essayant de conserver les
   ressemblances entre les lignes ;
4. **Clustering** : on regroupe automatiquement les lignes qui se ressemblent ;
5. **Visualisation** : on dessine le résultat.

## 1. Des données à beaucoup de dimensions

Pour commencer, fabriquons un fichier CSV de 1000 lignes et 300 colonnes, où chaque cellule est un nombre entier
aléatoire entre -1000 et 1000. Par convention, les noms de variables en MAJUSCULES désignent des **constantes** : des
paramètres que l'on fixe au début du programme et qu'on ne modifie plus.

In [ ]:
import random
import os
import csv

NOM_FICHIER = "data"
COLONNES = 300
LIGNES = 1000
MINMAX = [-1000, 1000]

tableau = []

for i in range(LIGNES):
  ligne = []
  for j in range(COLONNES):
    ligne.append(random.randint(MINMAX[0], MINMAX[1]))
  tableau.append(ligne)

# On n'imprime que le début de la première ligne : le tableau entier contient 300 000 nombres !
print(tableau[0][:10])

with open(os.path.join(os.getcwd(), f"{NOM_FICHIER}.csv"), "w", encoding="utf-8", newline="") as f:
  writer = csv.writer(f)
  for ligne in tableau:
    writer.writerow(ligne)

### Charger les données avec numpy

scikit-learn travaille avec des **tableaux numpy** (`ndarray`). [numpy](https://numpy.org/) est le package de base du
calcul scientifique en Python : ses tableaux ressemblent à des listes de listes, mais ne contiennent que des nombres, et
les calculs dessus sont beaucoup plus rapides. La fonction `np.genfromtxt()` lit directement un CSV de nombres dans un
tableau numpy.

L'attribut `.shape` donne la forme du tableau : (nombre de lignes, nombre de colonnes).

In [ ]:
import numpy as np

chemin = os.path.join(os.getcwd(), f"{NOM_FICHIER}.csv")
raw_data = np.genfromtxt(chemin, delimiter=",")

print(raw_data)
print(type(raw_data))
print(raw_data.shape)

Avec numpy, on peut sélectionner une colonne entière avec `tableau[:, numero]` (le `:` veut dire « toutes les
lignes ») : c'est plus pratique que de parcourir les lignes une par une.

## 2. La réduction de dimension avec t-SNE

Chaque ligne de notre tableau est un point dans un espace à 300 dimensions : impossible à dessiner ! La **réduction de
dimension** cherche à placer chaque point dans un espace à 2 dimensions, de manière à ce que les points qui se
ressemblaient (qui étaient proches dans l'espace à 300 dimensions) restent proches sur le dessin.

Nous utilisons l'algorithme [t-SNE](https://fr.wikipedia.org/wiki/Algorithme_t-SNE). Tous les outils de scikit-learn
s'utilisent de la même manière :
1. on crée un objet, avec ses paramètres (ici `n_components=2` : on veut 2 dimensions à l'arrivée) ;
2. on appelle sa méthode `fit_transform()` sur les données : l'algorithme « apprend » à partir des données (*fit*), puis
   les transforme (*transform*).

Le calcul peut prendre quelques secondes.

In [ ]:
from sklearn.manifold import TSNE

reducteur = TSNE(n_components=2, random_state=0)
reduit = reducteur.fit_transform(raw_data)

print(reduit.shape)
print(reduit)

In [ ]:
import matplotlib.pyplot as plt

# reduit[:, 0] est la première colonne (les x), reduit[:, 1] la deuxième (les y) :
plt.scatter(reduit[:, 0], reduit[:, 1])
plt.title("t-SNE des données brutes")
plt.show()

On obtient une sorte de nuage uniforme : c'est normal, nos données sont purement aléatoires, il n'y a aucune
ressemblance particulière entre les lignes !

> 💡 Il existe d'autres méthodes de réduction de dimension dans scikit-learn : `PCA` (analyse en composantes
> principales, plus simple et plus rapide), ou `UMAP` (dans le package `umap-learn`). Elles s'utilisent de la même
> manière.

## 3. Le prétraitement : mettre les données à l'échelle

Beaucoup d'algorithmes sont sensibles à l'**échelle** des données. Imaginez un tableau avec une colonne « âge » (entre
18 et 90) et une colonne « revenu » (entre 0 et 100 000) : sans précaution, le revenu écraserait complètement l'âge dans
le calcul des ressemblances. On transforme donc les colonnes pour qu'elles soient comparables. Deux méthodes courantes :
- `MinMaxScaler` ramène chaque colonne entre un minimum et un maximum (par défaut, entre 0 et 1) ;
- `StandardScaler` **standardise** chaque colonne : il la centre sur 0, avec un écart-type de 1. C'est la méthode la plus
  utilisée avant une réduction de dimension ou un clustering.

Ils s'utilisent comme t-SNE, avec `fit_transform()`. Ici, par exemple, on remet le résultat de t-SNE entre 0 et 1 :
regardez les axes, seule l'échelle a changé.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

normalise = MinMaxScaler((0, 1)).fit_transform(reduit)

plt.scatter(normalise[:, 0], normalise[:, 1])
plt.title("Après MinMaxScaler")
plt.show()

Et voici la version correcte du pipeline : on standardise les données **avant** la réduction de dimension.

In [ ]:
standardise = StandardScaler().fit_transform(raw_data)
print(standardise[0][:5])
print("Moyenne de la première colonne :", round(standardise[:, 0].mean(), 3))
print("Écart-type de la première colonne :", round(standardise[:, 0].std(), 3))

reduit = TSNE(n_components=2, random_state=0).fit_transform(standardise)

plt.scatter(reduit[:, 0], reduit[:, 1])
plt.title("t-SNE des données standardisées")
plt.show()

## 4. Le clustering avec KMeans

Le **clustering** consiste à regrouper automatiquement les points qui se ressemblent, sans leur donner d'étiquette au
préalable (on parle d'apprentissage **non supervisé**). L'algorithme le plus connu est
[KMeans](https://fr.wikipedia.org/wiki/K-moyennes) (k-moyennes) : on lui donne le nombre de groupes (*clusters*) à
trouver, et il cherche le meilleur découpage.

Après `fit()`, l'attribut `.labels_` contient le numéro du cluster attribué à chaque point.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=5, n_init=10, random_state=0)
km.fit(reduit)

print(km.labels_)
print(len(km.labels_), "étiquettes, une par point")

Pour visualiser les clusters, on colore chaque point selon son étiquette. Avec `plt.scatter()`, c'est très simple :
l'argument `c` accepte une liste de nombres, et attribue automatiquement une couleur à chaque valeur. `cmap` choisit la
palette de couleurs.

In [ ]:
plt.scatter(reduit[:, 0], reduit[:, 1], c=km.labels_, cmap="tab10")
plt.title("Les 5 clusters trouvés par KMeans")
plt.colorbar(label="Cluster")
plt.show()

⚠ **Attention** : KMeans a bien trouvé 5 clusters... alors que nos données sont purement aléatoires et ne contiennent
aucun groupe ! KMeans découpe toujours les données en autant de groupes qu'on lui demande, qu'ils existent vraiment ou
non. C'est une leçon importante : un algorithme donne toujours une réponse, c'est à nous d'avoir un regard critique
sur son résultat.

## Pour aller plus loin

- Dans le notebook d'exercices, vous appliquerez ce pipeline à des données qui contiennent de **vrais** groupes cachés :
  [Atelier Python Session 7 - Exercices.ipynb](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-7/Atelier%20Python%20Session%207%20-%20Exercices.ipynb).
- Pour appliquer ce pipeline à des textes, il faut d'abord les **vectoriser** : `TfidfVectorizer` (dans
  `sklearn.feature_extraction.text`) transforme un corpus en tableau de fréquences de mots, que l'on peut ensuite réduire
  et regrouper. Vous pourrez ainsi voir quels documents de votre corpus se ressemblent !